# U1-L4 实验：头部、步态与预定义动作

**实验目标**

认识并体验 K1 的头部控制、步态切换与预定义动作库：用 `set_head_angle` 控制头部俯仰与转向，`set_gait` 切换步态，`do_action` 调用厂商预置动作，理解头部视野与运动、步态对行走的影响。

> ⚠️ **安全提示**：头部控制、步态切换、预定义动作均涉及机器人运动或姿态变化：
> - 控制头部须进入 `walk` 模式（`set_head_angle` 仅在 walk 模式有效）。
> - 预定义动作会使机器人做出挥手、鞠躬等行为，运行前确认周围空旷、机器人处于稳定站立状态、教师监督。
> - 停止 / 收尾同样遵循 L2 安全规范：涉及行走时 `walk → prepare`，严禁 `walk → damping`。

> 📌 **前置准备**：本实验需在 K1 已完成开机、配网（连同一 WiFi）、查看 IP，并能用 BoosterStudio 成功连接之后进行；且学生已掌握 L1 信息读取与 L2 模式安全。

## 步骤 1：连接 K1 机器人

运行下面单元格建立连接。

In [1]:
from boosteros.robots.booster import BoosterRobot

robot = BoosterRobot()
print("已连接 K1 🤖")

[Booster OS] [boosteros.sdk] [INFO] SDK Channel initialized on default
[Booster OS] [boosteros.sdk] [INFO] Starting Booster SDK Local Adapter (mode=local, network_interface=default, virtual_robot_name=default).
[Booster OS] [boosteros.sdk] [INFO] Booster SDK Local Adapter started (mode=local, network_interface=default).


Network interface not specified. Will use environment variable (FASTRTPS_DEFAULT_PROFILES_FILE).
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024


[Booster OS] [boosteros.robot] [INFO] Fetching camera configuration from API...
[Booster OS] [boosteros.robot] [INFO] Using camera 'head' from API.
[Booster OS] [boosteros.BoosterHandGuidingManager] [INFO] BoosterHandGuidingManager initialized with default_record_file=/opt/booster/Gait/bin/joint_position.txt, record_archive_dir=~/.boosteros/hand_guiding
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_ad26f381] Booster ROS 2 Adapter initialized.
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_ad26f381] Booster ROS 2 Adapter started with 4 threads.
[Booster OS] [boosteros.robot] [INFO] Initializing audio manager & speech runtime...
[Booster OS] [boosteros.BoosterSpeechRuntime] [INFO] Booster speech runtime initialized for robot_name=''


ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024


[Booster OS] [boosteros.BoosterAudioManager] [INFO] BoosterAudioManager initialized
[Booster OS] [boosteros.robot] [INFO] Booster OS resources initialized.


ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
已连接 K1 🤖


## 步骤 2：认识动作库

复习 L1 学到的 `list_actions()`，认识可调用预定义动作及其 `id`——`do_action` 正是通过该 `id` 指定要执行哪个动作。

In [7]:
print("当前运行模式：", robot.get_mode())
actions = robot.list_actions()
for a in actions:
    print(f"id={a.id}  type={a.type}  时长={a.duration}s  可打断={a.interruptible}")

当前运行模式： damping
id=hand_shake  type=upper_body  时长=-1.0s  可打断=True
id=hand_wave  type=upper_body  时长=-1.0s  可打断=True
id=dance_new_year  type=upper_body  时长=-1.0s  可打断=True
id=dance_rock_rolling  type=upper_body  时长=-1.0s  可打断=True
id=dance_towards_future  type=upper_body  时长=-1.0s  可打断=True
id=gesture_dabing  type=upper_body  时长=-1.0s  可打断=True
id=gesture_ultraman  type=upper_body  时长=-1.0s  可打断=True
id=bow  type=upper_body  时长=-1.0s  可打断=True
id=cheer  type=upper_body  时长=-1.0s  可打断=True
id=lucky_cat  type=upper_body  时长=-1.0s  可打断=True


## 步骤 3：头部控制（set_head_angle）

头部姿态由两个角度控制，单位均为**弧度**：

- `pitch`：俯仰角，**正值 = 低头（看下方）**，负值 = 抬头
- `yaw`：偏航角，**正值 = 头向左转**，负值 = 头向右转
- 物理限位约：`pitch` -0.3 ~ 1.0，`yaw` -0.785 ~ 0.785

`set_head_angle` **仅在 `walk` 模式下有效**；在 damping / prepare 下调用会报错。下面按「每步独立 cell + 人工确认窗口」拆分，代码已加注释，仅在安全环境、教师监督下取消注释运行。

In [ ]:
# —— 第 1 步：进入站立准备态 ——
# robot.set_mode("prepare")      # 确保处于站立准备态（此时机器人尚未被拉起）

<font color="red">确定已经手动拉起机器人、处于直立站立、周围空旷、教师监督下，再运行下一格。</font>

In [ ]:
# —— 第 2 步：进入行走态（头部控制仅在 walk 有效）——
# robot.set_mode("walk")         # 进入行走态（须已 STAND 直立站稳）

### 3.1 头部角度体验

在 `walk` 模式下，分别体验：低头、转头、回正（每格代码取消注释后运行）。
1. 低头：

In [ ]:
# robot.set_head_angle(0.3, 0)    # 低头约 0.3 rad

2. 转头

In [ ]:
# robot.set_head_angle(0, 0.5)    # 头向左转 0.5 rad

3. 回正

In [ ]:
# robot.set_head_angle(0, 0)      # 回正

In [ ]:
# —— 头部体验结束，回到站立稳定 ——
# robot.set_mode("prepare")      # 安全停止：回到站立稳定，禁止直接切 damping

## 步骤 4：步态切换（set_gait）

K1 支持多种步态，常用两种：

- `"default"`：标准行走步态
- `"soccer"`：足球步态（用于踢球等动作与运动）

`set_gait` 有个方便特性：**若当前不在 `walk` 模式，设置会被记录，并在下次进入 `walk` 时生效**；若已在 `walk` 模式则立即切换。可先安全查看当前步态。

In [ ]:
print("支持的步态：", robot.list_gaits())
print("当前步态：", robot.get_gait())
robot.set_gait("soccer")          # 不在 walk 时仅记录，下次进入 walk 生效
print("设置后步态：", robot.get_gait())

实际体验两种步态的差异，需要在 `walk` 下进行。

### 4.1 步态体验

切换为足球步态，完成一段前进行走。

In [ ]:
import time

# robot.set_gait("soccer")       # 切换为足球步态（行走中观察差异）
# robot.set_velocity(0.5, 0, 0)  # 前进 0.5 m/s
# time.sleep(2)
# robot.set_velocity(0, 0, 0)    # 停止

再切回标准步态，完成一段前进行走。

In [ ]:
# robot.set_gait("default")      # 切回标准步态
# robot.set_velocity(0.5, 0, 0)  # 前进 0.5 m/s
# time.sleep(2)
# robot.set_velocity(0, 0, 0)    # 停止


最后回到站立稳定。

In [ ]:
# robot.set_mode("prepare")      # 安全停止：回到站立稳定

## 步骤 5：预定义动作（do_action）

`do_action(action_id)` 调用一个厂商预置动作，参数 `action_id` 来自 `list_actions()` 的 `id`（如挥手、鞠躬等）。

> ⚠️ 运行前确认：机器人处于稳定站立状态（prepare 已 STAND，或 walk 静止），周围空旷、教师监督；若动作未执行，先确认已 STAND 站立。

下面依次执行前 3 个预定义动作，每个动作独立一格，便于观察。

In [ ]:
import time
actions = robot.list_actions()
print("将依次执行前 3 个预定义动作，共", min(3, len(actions)), "个")

### 5.1 第 1 个动作

In [ ]:
# robot.do_action(actions[0].id)
# time.sleep(4)   # 等待动作完成

### 5.2 第 2 个动作

In [ ]:
# robot.do_action(actions[1].id)
# time.sleep(4)

### 5.3 第 3 个动作

In [ ]:
# robot.do_action(actions[2].id)
# time.sleep(4)

## 步骤 6：综合与安全收尾

- 头部、步态、预定义动作都是对“已站立机器人”的能力扩展；把它们组合起来，可形成“边走边转头扫描”等行为。
- 本单元三个实验（L2 模式、L3 运动、L4 头/步态/动作）都采用**开环**方式：发出指令后观察结果，不自动纠正偏差。
- 结束实验时，确保机器人回到稳定状态：`walk → prepare` 回到站立，或坐下后进入 `damping`；严禁 `walk → damping` 直接切换。

## 学习评价

请独立完成以下题目，检验本实验的掌握情况。

**一、判断题**（对的打√，错的打×）

1. `set_head_angle(pitch, yaw)` 只能在 `walk` 模式下调用，在 damping / prepare 下调用会报错。（    ）

**二、填空题**

2. 切换步态用 `set_gait(____)`；若当前不在 `walk` 模式，该设置会 ______（填“立即生效”或“记录并在下次进入 walk 时生效”）。

**三、选择题**（单选）

3. 下列接口中，用于在 walk 模式下控制机器人头部姿态的是（    ）。
   A. `set_velocity`　　B. `set_gait`　　C. `set_head_angle`　　D. `do_action`

> 参考答案（教师留存，勿印发给学生）：
> 1. √　2. `"soccer"` 或 `"default"`（任写其一即可）；记录并在下次进入 walk 时生效　3. C